Preprocessing for meta parameter

In [1]:
import numpy as np
import os
import pydicom
import dicom_to_zarr
import helpers
import zarr
import mdreg
import time
from mdreg import fit_models, elastix, skimage, ants, io
import dask.array as da

In [2]:
# Define paths (please adjust if needed)
dicom_folder = '/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/Raw_Datasets/159269_B1/15/DICOM' 
zarr_file = '/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/Raw_Datasets/159269_B1/14/159269_B1_15.zarr'
file_path = '/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/Raw_Datasets/159269_B1/15/AIF.txt'
results_path = '/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/MRI-Datasets/mdreg_motion_correction_results/159269_B1_15/'

In [3]:
try:
    # Lade die erste (Index 0) und zweite (Index 1) Spalte
    tacq, aif = np.loadtxt(
        file_path, 
        skiprows=3,       
        usecols=(0, 1),   
        unpack=True         
    )

    print("Daten erfolgreich geladen!")
    print(f"Form des AIF-Arrays: {aif.shape}")
    print(f"Form des Zeit-Arrays: {tacq.shape}")
    print("\nErste 5 Werte des AIF:")
    print(aif[:5])
    print("\nErste 5 Werte der Zeitachse (in Sekunden):")
    print(tacq[:5])

except FileNotFoundError:
    print(f"Fehler: Die Datei '{file_path}' wurde nicht gefunden.")
except Exception as e:
    print(f"Ein Fehler ist aufgetreten: {e}")


Daten erfolgreich geladen!
Form des AIF-Arrays: (250,)
Form des Zeit-Arrays: (250,)

Erste 5 Werte des AIF:
[447.205  396.251  395.1088 415.5565 422.3054]

Erste 5 Werte der Zeitachse (in Sekunden):
[0.000000e+00 9.918213e-05 1.983643e-04 2.975464e-04 3.967285e-04]


In [4]:
aif_list = aif.tolist()
tacq_list = tacq.tolist()

In [5]:
# Make sure the folder exists before calling the function
if not os.path.exists(dicom_folder):
    os.makedirs(dicom_folder)
    print(f"Folder '{dicom_folder}' was created. Please fill it with your DICOM files.")
elif not os.listdir(dicom_folder):
    print(f"The folder '{dicom_folder}' is empty. Please add DICOM files to run the script.")
else:
    # Call the conversion function
    dicom_to_zarr.convert_dicom_to_zarr(dicom_folder, zarr_file)

Lese DICOM-Dateien aus: /mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/Raw_Datasets/159269_B1/15/DICOM
Stapele DICOM-Schichten zu einem 3D-Raum...
Extrahierte Metadaten: {'PixelSpacing': [1.9531, 1.9531], 'SliceThickness': 5.0, 'Rows': 256, 'Columns': 256, 'PatientID': 'MEDCIC_10_B1'}
Speichere Array der Größe (12500, 256, 256) in /mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/Raw_Datasets/159269_B1/14/159269_B1_15.zarr...
Konvertierung abgeschlossen!


In [7]:
def fit_parallel(moving,
        fit_pixels = None,
        fit_coreg = None,
        fit_image = None,
        tol = 1e-6,    
        maxit = 5,
        verbose = 0,
        force_2d = False,
        path = None, 
    ):
    """
    Remove motion from a series of 2D- or 3D images.

    Parameters
    ----------
    moving : numpy.ndarray | zarr.Array
        The series of images to be corrected, with dimensions (x,y,t) or (x,y,z,t). 
    fit_pixels : dict, optional
        A dictionary defining a single-pixel signal model. The possible items 
        in the dictionary are the keywords of the function `mdreg.fit_pixels`. 
        For a slice-by-slice computation (4D array with force_2d=True), 
        *fit_pixels* can be a list of dictionaries, one for each slice. 
        The default is None.
    fit_coreg : dict, optional
        The parameters for coregistering the images. *fit_coreg* has one 
        required item 'package' with possible values 'skimage' (default), 
        'elastix' and 'ants'. The other parameters are the possible keywords 
        of the *coreg_series* function of the package specified. 
    fit_image : dict or list, optional
        A dictionary defining the function to fit the signal data, and its 
        parameter values. This argument is ignored if *fit_pixels* is already 
        provided. *fit_image* has one required key 'func' that specifies the 
        fit function to use. The other entries are the keyword arguments of 
        this fit function. 
        The fit function can be one of the functions built in to mdreg, or a 
        custom made function. A valid fit function *must* take a signal array 
        as argument, and return two variables: an array with the same shape 
        containing the fit to the model, and a second variable that contains 
        the fitted parameters. 
        For a slice-by-slice computation (4D array with force_2d=True), 
        *fit_image* can be a list of dictionaries, one for each slice. 
        If *fit_image* is not provided, a constant model is used. 
    tol : float, optional
        Stopping criterion for the iteration. The iteration stops if the 
        largest difference between new and old coregistered series in any 
        pixel at any time point is less than *tol* of the largest value. 
        The default is 1e-6.
    maxit : int, optional
        The maximum number of iterations. The default is 0.
    verbose : int, optional
        The level of feedback to provide to the user. 0: no feedback; 1: text 
        output only; 2: text output and progress bars. The default is 2.
    force_2d : bool, optional
        By default, a 3-dimensional moving array will be coregistered with a 
        3-dimensional deformation field. To perform slice-by-slice 
        2-dimensional registration instead, set *force_2d* to True. This 
        keyword is ignored when the arrays are 2-dimensional. The 
        default is False.
    path : str, optional
        Path on disk where to save the results. If no path is provided, the 
        results are not saved to disk. Defaults to None.

    Returns
    -------
    coreg : numpy.ndarray | zarr.Array
        The coregistered images with the same dimensions as *moving*.
    fit : numpy.ndarray | zarr.Array
        The fitted signal model with the same dimensions as *arr*.
    transfo : numpy.ndarray | zarr.Array | list
        The parameters of the transformation deforming the moving image to the 
        coregistered image. With skimage, this is the deformation field with 
        the same dimensions as *moving*, and one additional dimension for the 
        components of the vector field. With elastix this is an array of 
        parameter objects and with ants this is an array of files with 
        transform parameters. Note when force_2d = True these are 2-dimensional 
        arrays with one transform per slice and per time point.
    pars : numpy.ndarray | zarr.Array
        The parameters of the fitted signal model with dimensions (x,y,n) or 
        (x,y,z,n), where n is the number of free parameters of the signal 
        model.
 
    """

    print("Test")
    
    # Set defaults in fit_coreg
    if fit_coreg is None:
        fit_coreg = {'package': 'skimage'}
    if 'package' not in fit_coreg:
        fit_coreg['package'] = 'skimage'
    #if 'progress_bar' not in fit_coreg:
        #fit_coreg['progress_bar'] = verbose>1
    if 'name' not in fit_coreg:
        fit_coreg['name'] = 'coreg'

    # 2D slice-by-slice coregistration
    if moving.ndim==4:
        if force_2d:
            return  _fit_force_2d(
               moving, fit_image, fit_coreg, fit_pixels, tol, maxit, 
               verbose, path, 
            )
        
    # Set defaults for fit_image  
    if fit_image is None:
        fit_image = {'func': fit_models.fit_constant}

    # Check inputs
    if not isinstance(fit_image, dict):
        raise ValueError('The fit_image argument must be a dictionary.')

    # Set paths    
    _set_path(fit_coreg, path)
    _set_path(fit_image, path)
    _set_path(fit_pixels, path)

    # Compute
    converged = False
    it = 1
    start = time.time()

    if verbose > 0:
        print('Initializing..')
    coreg = io._copy(moving, path, 'coreg')

    while not converged: 

        startit = time.time()

        # Fit signal model
        if verbose > 0:
            print(f'Iteration {it}: fitting signal model')
        if fit_pixels is not None:
            fit, pars = fit_models.fit_pixels(coreg, **fit_pixels)
        else:
            kwargs = {i:fit_image[i] for i in fit_image if i!='func'}
            fit, pars = fit_image['func'](coreg, **kwargs)
        
        # Fit deformation
        if verbose > 0:
            print(f'Iteration {it}: fitting deformation fields')
        coreg_curr = io._copy(coreg, path, 'tmp')
        vals = _coreg_series(moving, fit, **fit_coreg)

        coreg, transfo = vals[:2]

        # Check convergence
        converged = _diff(coreg, coreg_curr) < tol
        
        if verbose > 0:
            print(f'Calculation time for iteration {it}: '
                  f'{(time.time()-startit)/60} min')  

        if it == maxit: 
            break

        it += 1 

    if verbose > 0:
        print(f'Total calculation time: {(time.time()-start)/60} min')

    io._remove(path, 'tmp')
    if len(vals) > 2: # optional return value
        defo = vals[2]
        return coreg, fit, transfo, pars, defo
    else:
        return coreg, fit, transfo, pars


def _fit_force_2d(
        moving, fit_image, fit_coreg, fit_pixels, tol, maxit, verbose, 
        path,
    ):

    # Required outputs
    coreg = io._copy(moving, path, 'coreg')
    if fit_coreg['package'] == 'skimage':
        transfo = io._defo(
            moving, 
            path, 
            force_2d=True, 
            name=fit_coreg['name']+'_defo',
        )
    else:
        transfo = np.empty(moving.shape[-2:], dtype=object)

    # Optional outputs
    defo = None
    if 'return_deformation' in fit_coreg:
        if fit_coreg['return_deformation']:
            defo = io._defo(
                moving, 
                path, 
                force_2d=True, 
                name=fit_coreg['name']+'_defo',
            )

    for k in tqdm(
            range(moving.shape[2]), 
            desc='Fitting slice', 
            disable=verbose<2,
        ):
        if verbose == 1:
            print(f'Fitting slice {k+1} / {moving.shape[2]}')

        if fit_image is None:
            fit_image_k = None
        elif isinstance(fit_image, dict):
            fit_image_k = fit_image
        else:
            fit_image_k = fit_image[k]

        if fit_pixels is None:
            fit_pixels_k = None
        elif isinstance(fit_pixels, dict):
            fit_pixels_k = fit_pixels
        else:
            fit_pixels_k = fit_pixels[k]

        vals = fit_parallel(
            moving[:,:,k,:],
            fit_pixels = fit_pixels_k,
            fit_image = fit_image_k,
            fit_coreg = fit_coreg,
            tol = tol,
            maxit = maxit,
            verbose = verbose,
        )
        coreg[:,:,k,:], fit_k, transfo_k, pars_k = vals[:4]
        if k == 0:
            fit_arr, pars = io._fit_models_init(moving, path, pars_k.shape[-1])              
        if fit_coreg['package'] == 'skimage':
            transfo[:,:,k,:,:] = transfo_k
        else:
            transfo[k,:] = transfo_k
        fit_arr[:,:,k,:] = fit_k
        pars[:,:,k,:] = pars_k
        if defo is not None:
            defo[:,:,k,:,:] = vals[4]
    if defo is None:
        return coreg, fit_arr, transfo, pars
    else:
        return coreg, fit_arr, transfo, pars, defo



def _set_path(dct, path):
    if dct is None:
        return
    if path is None:
        return
    if 'path' in dct:
        if path != dct['path']:
            raise ValueError("Two different paths are provided.")
    else:
        dct['path'] = path
        

def _diff(coreg, coreg_curr):
    if isinstance(coreg, np.ndarray):
        corr = np.max(np.abs(coreg-coreg_curr))/np.max(np.abs(coreg_curr))
    else:
        coreg = da.from_zarr(coreg) 
        coreg_curr = da.from_zarr(coreg_curr)    
        corr = da.max(da.abs(coreg-coreg_curr))/da.max(da.abs(coreg_curr))
        corr.compute()
    return corr


def _coreg_series(moving, fit, package='elastix', **fit_coreg):

    if package == 'elastix':
        print('elastix')
        fit_coreg = _set_mdreg_elastix_defaults(fit_coreg)
        return elastix.coreg_series(moving, fit, **fit_coreg)
    
    elif package == 'skimage':
        print("parallelization")
        return skimage.coreg_series(moving, fit, **fit_coreg, parallel=True, progress_bar=False)
    
    elif package == 'ants':
        print('ants')
        return ants.coreg_series(moving, fit, **fit_coreg)
    
    else:
        raise NotImplementedError(
            'This coregistration package is not implemented')
    

def _set_mdreg_elastix_defaults(params):

    if "WriteResultImage" not in params:
        params["WriteResultImage"] = "false"
    if "WriteDeformationField" not in params:
        params["WriteDeformationField"] = "false"
    if "ResultImagePixelType" not in params:
        params["ResultImagePixelType"] = "float"

    # # Removing this for v0.4.2 as results appear to be worse
    # if 'Metric' not in params:
    #     params["Metric"] = "AdvancedMeanSquares"

    # # Settings pre v0.4.0 - unclear why - removed for now
    # if "FinalGridSpacingInPhysicalUnits" not in params:
    #     params["FinalGridSpacingInPhysicalUnits"] = "50.0"
    # if "AutomaticParameterEstimation" not in params:
    #     params["AutomaticParameterEstimation"] = "true"
    # if "ASGDParameterEstimationMethod" not in params:
    #     params["ASGDParameterEstimationMethod"] = "Original"
    # if "MaximumStepLength" not in params:
    #     params["MaximumStepLength"] = "1.0"
    # if "CheckNumberOfSamples" not in params:
    #     params["CheckNumberOfSamples"] = "true"
    # if "RandomCoordinate" not in params:
    #     params["ImageSampler"] = "RandomCoordinate"

    return params

In [8]:
data = zarr.open(zarr_file)
dask_array = da.from_array(data)
dask_array

dask.array<array, shape=(12500, 256, 256), dtype=int16, chunksize=(1024, 256, 256), chunktype=numpy.ndarray>

In [9]:
temp_zarr = 'input_zarr_for_mdreg.zarr'

In [10]:
import shutil
dask_array = np.transpose(dask_array, [1,2,0])
dask_array = dask_array.reshape([256,256,50,250])
dask_array = np.transpose(dask_array, [0,1,2,3])
if os.path.exists(temp_zarr):
    shutil.rmtree(temp_zarr)
dask_array.to_zarr(temp_zarr)

In [11]:
temp_zarr = zarr.open('input_zarr_for_mdreg.zarr')
temp_zarr

<Array file://input_zarr_for_mdreg.zarr shape=(256, 256, 50, 250) dtype=int16>

In [10]:
temp = temp_zarr[:,:,25,:]
temp = np.transpose(temp, [2,0,1])
temp

array([[[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       ...,

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 

In [11]:
helpers.explore_3D_array(temp)

interactive(children=(IntSlider(value=124, description='SLICE', max=249), Output()), _dom_classes=('widget-int…

In [12]:
# Path for output
results_path = results_path
aif = aif
coreg, fit, transfo, pars = fit_parallel(
    temp_zarr,
    fit_image={
        'func': mdreg.fit_2cm_lin,
        'time': tacq_list, # Acquisition times
        'aif': aif_list,   # Signal-time curve in the aorta
        'baseline': 1,# Nr of precontrast samples
    },
    fit_coreg={'package':'skimage',},
    maxit=3,
    path=results_path,
    verbose=1,
)

Test
Initializing..
Iteration 1: fitting signal model


Fitting 2cm: 100%|██████████| 50/50 [05:59<00:00,  7.20s/it]


Iteration 1: fitting deformation fields
parallelization


RuntimeError: Zstd decompression error: b'Src size is incorrect'

In [ ]:
# Path for output
results_path = results_path
aif = aif
coreg, fit, transfo, pars = fit_parallel(
    temp_zarr,
    fit_image={
        'func': mdreg.fit_2cm_lin,
        'time': tacq, # Acquisition times
        'aif': aif,   # Signal-time curve in the aorta
        'baseline': 1,# Nr of precontrast samples
    },
    fit_coreg={'package':'ants',},
    maxit=3,
    path=results_path,
    verbose=1,
)

Test
Initializing..
Iteration 1: fitting signal model


Fitting 2cm: 100%|██████████| 50/50 [05:10<00:00,  6.22s/it]


Iteration 1: fitting deformation fields
ants


RuntimeError: Zstd decompression error: b'Src size is incorrect'

In [ ]:
# Path for output
results_path = results_path
aif = aif
coreg, fit, transfo, pars = fit_parallel(
    temp_zarr,
    fit_image={
        'func': mdreg.fit_2cm_lin,
        'time': tacq, # Acquisition times
        'aif': aif,   # Signal-time curve in the aorta
        'baseline': 1,# Nr of precontrast samples
    },
    fit_coreg={'package':'elastix',},
    maxit=3,
    path=results_path,
    verbose=1,
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def find_baseline_from_aif(time_vector, aif_curve, initial_baseline_points=5, threshold_std=3):
    """
    Ermittelt und plottet die Baseline automatisch aus einer AIF-Kurve.

    Args:
        time_vector (np.ndarray): Der 1D-Array mit den Zeitpunkten.
        aif_curve (np.ndarray): Der 1D-Array mit den Signalintensitäten (der AIF).
        initial_baseline_points (int): Anzahl der ersten Punkte, die sicher zur Baseline gehören.
        threshold_std (int): Anzahl der Standardabweichungen, die das Signal überschreiten muss,
                             um als Anstieg erkannt zu werden.

    Returns:
        int: Die ermittelte Baseline (Anzahl der Punkte vor dem Anstieg).
    """
    if len(aif_curve) < initial_baseline_points + 2:
        print("WARNUNG: AIF-Kurve ist zu kurz für eine automatische Bestimmung.")
        return initial_baseline_points

    # 1. Berechne Statistik der initialen Baseline
    initial_baseline_signal = aif_curve[:initial_baseline_points]
    baseline_mean = np.mean(initial_baseline_signal)
    baseline_std = np.std(initial_baseline_signal)
    
    # Definiere die Schwelle für den Signalanstieg
    upper_threshold = baseline_mean + threshold_std * baseline_std

    # 2. Finde den ersten Punkt, der die Schwelle überschreitet
    # Wir starten die Suche nach der initialen Baseline-Periode
    baseline_end_index = -1
    for i in range(initial_baseline_points, len(aif_curve)):
        if aif_curve[i] > upper_threshold:
            baseline_end_index = i
            break
            
    # Fallback, falls kein Anstieg gefunden wird (z.B. bei reinen Pre-Contrast-Daten)
    if baseline_end_index == -1:
        print("WARNUNG: Kein signifikanter Signalanstieg gefunden.")
        baseline_end_index = initial_baseline_points

    # Die Baseline ist die Anzahl der Punkte VOR dem Anstieg
    detected_baseline = baseline_end_index

    # 3. Erstelle den Graphen zur visuellen Überprüfung
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.figure(figsize=(12, 6))
    
    # Plotte die komplette AIF-Kurve
    plt.plot(time_vector, aif_curve, marker='o', linestyle='-', label='AIF Signal', zorder=2)
    
    # Plotte die berechnete Baseline-Schwelle
    plt.axhline(y=upper_threshold, color='red', linestyle='--', label=f'Schwelle ({threshold_std} Std. Abw.)', zorder=1)
    
    # Markiere das Ende der Baseline mit einer vertikalen Linie
    plt.axvline(x=time_vector[detected_baseline], color='green', linestyle='-', linewidth=2, label=f'Ende der Baseline (Index {detected_baseline})', zorder=3)
    
    # Schattiere den Baseline-Bereich
    plt.axvspan(time_vector[0], time_vector[detected_baseline], color='green', alpha=0.1, label='Baseline-Region')

    plt.title('Automatische Baseline-Ermittlung aus AIF', fontsize=16)
    plt.xlabel('Zeit (Sekunden)', fontsize=12)
    plt.ylabel('Signalintensität', fontsize=12)
    plt.legend()
    plt.grid(True)
    plt.show()

    return detected_baseline

# ==================================================================
# ANWENDUNG
# ==================================================================
# --- Erstelle Dummy-Daten, die eine typische AIF-Kurve simulieren ---
# (Ersetzen Sie diese Zeilen durch das Laden Ihrer echten AIF-Datei)
num_points = 100
tacq = np.linspace(0, 50, num_points) # 50 Sekunden, 100 Messpunkte

# Erzeuge eine realistische AIF-Kurve
baseline_level = 250
noise = np.random.normal(0, 5, num_points)
bolus = 800 * np.exp(-((tacq - 15)**2) / 4) # Gauß-Peak bei t=15s
washout = -150 * (1 - np.exp(-(tacq - 15) / 10))
washout[tacq < 15] = 0

# Dies ist Ihre AIF-Kurve
aif = baseline_level + noise + bolus + washout
aif[:5] = baseline_level + np.random.normal(0, 5, 5) # Sorge für eine saubere initiale Baseline
# --- Rufe die Funktion mit den AIF-Daten auf ---
# Sie können 'threshold_std' anpassen, falls die Erkennung zu früh/spät ist
ermittelte_baseline = find_baseline_from_aif(tacq, aif, threshold_std=10)

print(f"\nAutomatisch ermittelte Baseline: {ermittelte_baseline}")
print("Dies ist die Anzahl der Zeitpunkte VOR dem Kontrastmittelanstieg.")